In [3]:
from pathlib import Path
import json
from pprint import pprint

from langchain_core.documents import Document

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter
)

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

load_dotenv()

True

# Configuration

In [4]:
DATA_PATH = Path("../data")

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

EMBEDDING_MODEL = "text-embedding-3-small"

VECTOR_DB_PATH = "../vector_store"

# Exploration des fichiers

In [5]:
for file in DATA_PATH.glob("*"):
    print(file.name)

corpus_faq_interne.md
corpus_guide_onboarding.md
corpus_note_produit_per.md
corpus_README.md
corpus_specs_api.md
corpus_tickets_support.json


# Fonction Markdown

In [6]:
def load_markdown_documents(folder_path):

    documents = []

    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3")
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )

    for md_file in Path(folder_path).glob("*.md"):

        with open(md_file, encoding="utf-8") as f:
            text = f.read()

        docs = markdown_splitter.split_text(text)

        for doc in docs:
            doc.metadata["source"] = md_file.name

        documents.extend(docs)

    return documents

# Fonction JSON

In [28]:
from pathlib import Path
import json
from langchain_core.documents import Document


def load_ticket_documents(json_file):

    json_file = Path(json_file)

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = []

    for ticket in data["tickets"]:

        content = f"""
ID : {ticket['id']}
Date : {ticket['date']}
Catégorie : {ticket['category']}
Statut : {ticket['status']}
Priorité : {ticket['priority']}

Sujet :
{ticket['subject']}

Description :
{ticket['description']}

Résolution :
{ticket['resolution']}

Agent :
{ticket['agent']}

Durée :
{ticket['duration_hours']} heures
"""

        documents.append(
            Document(
                page_content=content.strip(),
                metadata={
                    "source": json_file.name,
                    "ticket_id": ticket["id"],
                    "category": ticket["category"]
                }
            )
        )

    return documents

In [14]:
ticket_docs = load_ticket_documents(
    "../data/corpus_tickets_support.json"
)

print(ticket_docs[0].page_content)


ID : TKT-2025-00112
Date : 2025-04-10
Catégorie : Retrait
Statut : resolved
Priorité : high

Sujet :
Retrait bloqué depuis 5 jours

Description :
J'ai effectué une demande de retrait partiel de 800€ sur mon LER le 05/04. La somme n'est toujours pas créditée sur mon compte bancaire et je n'ai reçu aucun mail de confirmation.

Résolution :
Le retrait était en attente de validation manuelle suite à un changement d'IBAN effectué moins de 72h avant la demande. Mesure de sécurité standard. Le virement a été libéré et crédité le 11/04. L'adhérent a été informé par email et téléphone.

Agent :
Sophie L.

Durée :
28 heures



# Chargement des données

In [29]:
md_docs = load_markdown_documents(DATA_PATH)

ticket_docs = load_ticket_documents(DATA_PATH/"corpus_tickets_support.json")

In [30]:

print(f"Markdown docs : {len(md_docs)}")
print(f"Tickets docs : {len(ticket_docs)}")

Markdown docs : 40
Tickets docs : 7


In [31]:
md_docs[0]

Document(metadata={'h1': 'FAQ Interne — Espace Épargne & Contrats LFM', 'h2': 'Général', 'source': 'corpus_faq_interne.md'}, page_content="**Q : Qu'est-ce que La France Mutualiste ?**\nLa France Mutualiste est une mutuelle d'épargne fondée en 1946, spécialisée dans les produits d'épargne retraite et de prévoyance pour les agents de la fonction publique et les particuliers. Elle appartient au groupe Malakoff Humanis depuis 2018.  \n**Q : Quels types de contrats proposez-vous ?**\nNous proposons principalement :\n- Le contrat Livret Épargne Retraite (LER) : contrat d'épargne à taux garanti\n- Le contrat MultiVie : assurance-vie multisupport\n- Le contrat Prévi Retraite : plan d'épargne retraite (PER individuel)\n- Le contrat Capital Décès : prévoyance  \n**Q : Comment contacter le service client ?**\n- Téléphone : 01 44 56 77 88 (lundi–vendredi, 9h–18h)\n- Email : contact@lfm.fr\n- Espace en ligne : monespace.lfm.fr\n- Courrier : 18 rue de Londres, 75009 Paris  \n---")

In [32]:
ticket_docs[0]

Document(metadata={'source': 'corpus_tickets_support.json', 'ticket_id': 'TKT-2025-00112', 'category': 'Retrait'}, page_content="ID : TKT-2025-00112\nDate : 2025-04-10\nCatégorie : Retrait\nStatut : resolved\nPriorité : high\n\nSujet :\nRetrait bloqué depuis 5 jours\n\nDescription :\nJ'ai effectué une demande de retrait partiel de 800€ sur mon LER le 05/04. La somme n'est toujours pas créditée sur mon compte bancaire et je n'ai reçu aucun mail de confirmation.\n\nRésolution :\nLe retrait était en attente de validation manuelle suite à un changement d'IBAN effectué moins de 72h avant la demande. Mesure de sécurité standard. Le virement a été libéré et crédité le 11/04. L'adhérent a été informé par email et téléphone.\n\nAgent :\nSophie L.\n\nDurée :\n28 heures")

# Chunking

In [33]:
#splitter = RecursiveCharacterTextSplitter(
    #chunk_size=CHUNK_SIZE,
    #chunk_overlap=CHUNK_OVERLAP
#)

#md_chunks = splitter.split_documents(md_docs)

# Fusion

In [34]:
all_docs = md_docs + ticket_docs

print(len(all_docs))

47


# Embedding

In [35]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Vector Store

In [36]:
vectorstore=Chroma.from_documents(documents=all_docs,embedding=embeddings,persist_directory=VECTOR_DB_PATH)

# Retriever

In [38]:
retriever = vectorstore.as_retriever(
    search_type="mmr", # MMR = Max Marginal Relevance, l'objectif par rapport à "similarity" c'est de garder des chunks pertinents mais différents 
                       #entre eux, pour éviter la redondance 
    search_kwargs={
        "k": 4 # 4 chunks diversifiés
    }
)

# Test Retrieval

In [41]:
docs = retriever.invoke(
    "Pourquoi un retrait peut être bloqué ?"
)

for doc in docs:

    print("="*100)
    pprint(doc.metadata)
    print(doc.page_content[:500])

{'category': 'Retrait',
 'source': 'corpus_tickets_support.json',
 'ticket_id': 'TKT-2025-00112'}
ID : TKT-2025-00112
Date : 2025-04-10
Catégorie : Retrait
Statut : resolved
Priorité : high

Sujet :
Retrait bloqué depuis 5 jours

Description :
J'ai effectué une demande de retrait partiel de 800€ sur mon LER le 05/04. La somme n'est toujours pas créditée sur mon compte bancaire et je n'ai reçu aucun mail de confirmation.

Résolution :
Le retrait était en attente de validation manuelle suite à un changement d'IBAN effectué moins de 72h avant la demande. Mesure de sécurité standard. Le virement
{'h1': 'Note Produit — PER Individuel "Prévi Retraite"',
 'h2': 'Mise à jour : Janvier 2025',
 'h3': '4. Cas de déblocage anticipé',
 'source': 'corpus_note_produit_per.md'}
Contrairement au PEA ou à l'assurance-vie, le PER est bloqué jusqu'à la retraite SAUF dans 6 cas :
1. Invalidité de 2e ou 3e catégorie (titulaire, conjoint, enfants)
2. Décès du conjoint ou partenaire pacsé
3. Expiration des dr

# Prompt

In [42]:
prompt = ChatPromptTemplate.from_template(
"""
Tu es un assistant interne de La France Mutualiste.

Réponds uniquement à partir du contexte fourni.

Si l'information n'existe pas dans les documents,
réponds :

"Je ne trouve pas cette information dans la documentation disponible."

Contexte :
{context}

Question :
{question}
"""
)

# LLM

In [43]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Fonction context

In [44]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

# Chaîne RAG

In [45]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test

In [49]:
response = rag_chain.invoke(
    "Comment se déroule la Gestion des réclamations ?"
)

print(response)

Toute réclamation doit être enregistrée dans Salesforce sous l'objet "Réclamation" dans les 24h suivant réception. La réponse au client doit intervenir sous 10 jours ouvrés, conformément au délai réglementaire ACPR.


# Citation des sources

In [47]:
question = "Pourquoi un retrait peut être bloqué ?"

docs = retriever.invoke(question)

sources = list(
    set(
        doc.metadata["source"]
        for doc in docs
    )
)

response = rag_chain.invoke(question)

print(response)

print("\nSources utilisées :")
print(sources)

Un retrait peut être bloqué en raison d'une validation manuelle, notamment si un changement d'IBAN a été effectué moins de 72 heures avant la demande de retrait. Cela constitue une mesure de sécurité standard.

Sources utilisées :
['corpus_specs_api.md', 'corpus_faq_interne.md', 'corpus_note_produit_per.md', 'corpus_tickets_support.json']


In [57]:
questions = [
    "Comment se déroule la Gestion des réclamations ?",
    "Comment obtenir un token ?",
    "Retrait bloqué depuis 5 jours",
    "Quelle est la météo à Paris?"
]

for question in questions:

    print("="*100)
    print(question)

    response = rag_chain.invoke(question)

    print(response)

Comment se déroule la Gestion des réclamations ?
Toute réclamation doit être enregistrée dans Salesforce sous l'objet "Réclamation" dans les 24h suivant réception. La réponse au client doit intervenir sous 10 jours ouvrés, conformément au délai réglementaire ACPR.
Comment obtenir un token ?
Pour obtenir un token, vous devez effectuer une requête POST à l'URL `/auth/token` avec les paramètres suivants dans le corps de la requête :

- `grant_type` : `client_credentials`
- `client_id` : votre identifiant client (`YOUR_CLIENT_ID`)
- `client_secret` : votre secret client (`YOUR_CLIENT_SECRET`)
- `scope` : les permissions requises (par exemple, `contracts:read operations:write`)

Le contenu de la requête doit être de type `application/x-www-form-urlencoded`. 

Après avoir envoyé la requête, vous recevrez une réponse contenant le `access_token`, le `token_type`, la durée d'expiration (`expires_in`), et le `scope` associé.
Retrait bloqué depuis 5 jours
Le retrait était en attente de validation

In [61]:
questions = [
    "Comment se déroule la Gestion des réclamations ?",
    "Comment obtenir un token ?",
    "Retrait bloqué depuis 5 jours",
    "Quelle est la météo à Paris?"
]

for question in questions:

    docs = retriever.invoke(question)

    response = rag_chain.invoke(question)

    sources = sorted(
        list(
            set(
                doc.metadata.get("source", "Unknown")
                for doc in docs
            )
        )
    )

    print("=" * 100)
    print(f"Question : {question}")
    print()
    print(f"Chunks récupérés : {len(docs)}")
    print(f"Sources : {sources}")
    print()
    print(response)


Question : Comment se déroule la Gestion des réclamations ?

Chunks récupérés : 4
Sources : ['corpus_faq_interne.md', 'corpus_guide_onboarding.md', 'corpus_specs_api.md']

Toute réclamation doit être enregistrée dans Salesforce sous l'objet "Réclamation" dans les 24h suivant réception. La réponse au client doit intervenir sous 10 jours ouvrés, conformément au délai réglementaire ACPR.
Question : Comment obtenir un token ?

Chunks récupérés : 4
Sources : ['corpus_README.md', 'corpus_guide_onboarding.md', 'corpus_specs_api.md']

Pour obtenir un token, vous devez effectuer une requête POST à l'URL `/auth/token` avec les paramètres suivants dans le corps de la requête :

- `grant_type=client_credentials`
- `client_id=YOUR_CLIENT_ID`
- `client_secret=YOUR_CLIENT_SECRET`
- `scope=contracts:read operations:write`

Assurez-vous que le `Content-Type` de la requête est `application/x-www-form-urlencoded`. Vous recevrez en réponse un JSON contenant l'`access_token`, le `token_type`, la durée d'ex